# Patient P2 — Session Selection and Preprocessing

## Objective

This notebook evaluates raw respiratory session files for Patient P2 and identifies physiologically suitable treatment sessions for inclusion in the variability analysis pipeline.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

from private_patient_mapping import public_filename

## Raw Respiratory Signal Loading

Functions for parsing ABC respiratory waveform `.dat` files and extracting usable signal data.

In [3]:
def load_raw(file_path):

    col_names = [
        "Time",
        "Volume",
        "Balloon_Valve",
        "Patient_Switch",
        "Gating_Mode",
        "Gating_Status",
        "Relay_State"
    ]

    start = None

    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if "HeaderEnd" in line:
                start = i + 1
                break

    # Skip files without valid ABC header
    if start is None:
        print(f"Skipping (no HeaderEnd): {public_filename(file_path)}")
        return None

    try:
        df = pd.read_csv(
            file_path,
            sep=r'\s*;\s*',
            skiprows=start,
            names=col_names,
            engine="python",
            na_values=["-", " - "]
        )
        return df

    except Exception as e:
        print(f"Error reading {public_filename(file_path)}: {e}")
        return None

## Signal Extraction and Cleaning

In [6]:
def extract_signal(df):
    df = df.dropna(subset=["Time", "Volume"])

    df["Time"] = df["Time"].astype(float)
    df["Volume"] = df["Volume"].astype(float)

    return df

## Respiratory Cycle Detection

Peak and trough detection is used to estimate physiological respiratory cycles from waveform data.

In [7]:
def detect_cycles(time, volume):

    # Detect peaks (inhale)
    peaks, _ = find_peaks(volume, distance=10)

    # Detect troughs (exhale)
    troughs, _ = find_peaks(-volume, distance=10)

    extrema = np.sort(np.concatenate([peaks, troughs]))

    cycles = []

    for i in range(len(extrema) - 2):
        i1, i2, i3 = extrema[i], extrema[i+1], extrema[i+2]

        duration = time[i3] - time[i1]
        amplitude = abs(volume[i2] - volume[i1])

        # Basic validity filter
        if duration > 0:
            cycles.append((duration, amplitude))

    return cycles

## Session Quality Evaluation

Sessions are filtered using duration and cycle-count criteria to exclude physiologically insufficient recordings.

In [8]:
def evaluate_signal(df):

    df = extract_signal(df)

    if len(df) < 50:
        return None

    time = df["Time"].values
    volume = df["Volume"].values

    duration = time[-1] - time[0]
    volume_std = np.std(volume)
    cycles = detect_cycles(time, volume)
    num_cycles = len(cycles)

    return {
        "duration": duration,
        "volume_std": volume_std,
        "num_cycles": num_cycles
    }

def evaluate_file_full(file_path):

    df = load_raw(file_path)

    if df is None:
        print(f"Rejected(no header): {public_filename(file_path)}")
        return None

    signal = evaluate_signal(df)

    if signal is None:
        print(f"Rejected(weak/short signal): {public_filename(file_path)}")
        return None

    # Physiological session inclusion criteria
    if signal["duration"] < 300:
        print(f"Rejected(too short): {public_filename(file_path)}")
        return None

    if signal["num_cycles"] < 100:
        print(f"Rejected(too few cycles): {public_filename(file_path)}")
        return None

    if signal["volume_std"] < 0.2:
        print(f"Rejected(low variability) {public_filename(file_path)}")
        return None

    print(f"Selected: {public_filename(file_path)}")

    return {
        "file": public_filename(file_path),
        "signal": signal
    }

## Session Selection Execution

In [9]:
def select_best_files(folder_path):
    selected = []

    for file in os.listdir(folder_path):
        if file.endswith(".dat"):
            path = os.path.join(folder_path, file)
            res = evaluate_file_full(path)

            if res:
                selected.append(res)

    return selected

select_best_files(".")

Selected: P2_1.dat
Selected: P2_boost 1.dat
Selected: P2_boost 2.dat
Selected: P2_boost 20.dat
Selected: P2_boost 21.dat
Selected: P2_boost 3.dat
Selected: P2_boost 4.dat
Selected: P2_ct...dat
Rejected(too short): P2_ct.dat
Selected: P2_ct1.dat
Rejected(too short): P2_TRT 01.dat
Selected: P2_Tx-10.dat
Selected: P2_Tx-11.dat
Rejected(too few cycles): P2_Tx-12.dat
Selected: P2_Tx-12_4_23_2024 11_35_48 AM.dat
Selected: P2_Tx-13.dat
Selected: P2_Tx-14.dat
Selected: P2_Tx-2.dat
Selected: P2_Tx-3..dat
Selected: P2_Tx-3.dat
Selected: P2_Tx-5..dat
Selected: P2_Tx-5.dat
Selected: P2_Tx-6.dat
Rejected(too few cycles): P2_Tx-6_4_13_2024 1_41_09 PM.dat
Rejected(too short): P2_Tx-6_4_13_2024 1_46_41 PM.dat
Selected: P2_Tx-7.dat
Selected: P2_Tx-8.dat
Selected: P2_Tx-9.dat


[{'file': 'P2_1.dat',
  'signal': {'duration': np.float64(522.22),
   'volume_std': np.float64(0.42968663610956004),
   'num_cycles': 324}},
 {'file': 'P2_boost 1.dat',
  'signal': {'duration': np.float64(522.22),
   'volume_std': np.float64(0.42968663610956004),
   'num_cycles': 324}},
 {'file': 'P2_boost 2.dat',
  'signal': {'duration': np.float64(2007.68),
   'volume_std': np.float64(0.42587177722632424),
   'num_cycles': 729}},
 {'file': 'P2_boost 20.dat',
  'signal': {'duration': np.float64(552.4599999999999),
   'volume_std': np.float64(0.4478458491533093),
   'num_cycles': 207}},
 {'file': 'P2_boost 21.dat',
  'signal': {'duration': np.float64(351.56),
   'volume_std': np.float64(0.5084107837310082),
   'num_cycles': 116}},
 {'file': 'P2_boost 3.dat',
  'signal': {'duration': np.float64(408.53999999999996),
   'volume_std': np.float64(0.4481995232115238),
   'num_cycles': 110}},
 {'file': 'P2_boost 4.dat',
  'signal': {'duration': np.float64(501.76),
   'volume_std': np.float64(

## Final Session Selection

Sessions were screened using recording duration, respiratory
cycle-count, and signal variability criteria to exclude short,
insufficiently sampled, or low-variability recordings.
Additional consistency validation was performed to identify
duplicate or repeated waveform files.

For Patient P2, five treatment sessions were retained for downstream
variability analysis:
- P2_Tx-3.dat
- P2_Tx-2.dat
- P2_Tx-14.dat
- P2_Tx-11.dat
- P2_Tx-5.dat

## Duplicate and Consistency Validation

Additional validation checks were performed to identify:
- duplicate respiratory recordings,
- filename inconsistencies,
- and repeated waveform sessions.

These checks were used to improve preprocessing reliability
before session selection.

In [10]:
def compute_signature(df):
    df = extract_signal(df)

    if len(df) < 10:
        return None

    time = df["Time"].values
    volume = df["Volume"].values

    return {
        "start_time": round(time[0], 2),
        "end_time": round(time[-1], 2),
        "length": len(df),
        "mean_vol": round(np.mean(volume), 4),
        "std_vol": round(np.std(volume), 4)
    }

def signature_key(sig):
    return (
        sig["start_time"],
        sig["end_time"],
        sig["length"],
        sig["mean_vol"],
        sig["std_vol"]
    )

def analyze_folder_consistency(folder_path):
    signatures = {}
    name_map = {}

    for file in os.listdir(folder_path):
        if not file.endswith(".dat"):
            continue

        path = os.path.join(folder_path, file)
        df = load_raw(path)

        if df is None:
            continue

        sig = compute_signature(df)
        if sig is None:
            continue

        key = signature_key(sig)

        # Check for identical respiratory recordings
        if key in signatures:
            print("Duplicate session detected:")
            print(f"   {public_filename(file)} == {public_filename(signatures[key])}")
        else:
            signatures[key] = file

        # Check for same filename consistency
        base_name = file.split(".dat")[0]

        if base_name in name_map:
            prev_sig = name_map[base_name]

            if signature_key(prev_sig) != key:
                print("Filename inconsistency detected:")
                print(f"   {base_name}")
        else:
            name_map[base_name] = sig

    print("\nFolder consistency check completed.")

In [11]:
analyze_folder_consistency(".")

Duplicate session detected:
   P2_boost 1.dat == P2_1.dat

Folder consistency check completed.


## Preprocessing Outcome

The selected treatment sessions passed duration, cycle-count, signal variability criteria
and consistency validation checks and were retained for
downstream respiratory variability analysis.